**Imports**

In [72]:
import re
import pandas as pd
import string
import numpy as np
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer

**Reading in Text File**

In [73]:
with open("TheGreatGatsby.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Splitting Chapters into Separate Chapters
chapters = re.split(r"\n\s*[IVX]+\s*\n", text)

# Cleaning Empty Entries
chapters = [ch.strip() for ch in chapters if len(ch.strip()) > 1000]

print(len(chapters))  # We will use first 9 since that is chapter count of the book

10


**The Cleaning Portion of the Text File to Organize**

In [ ]:
stopwords = {
    "the", "of", "a", "and", "to", "in", "that", "it", "was",
    "he", "she", "i", "you", "his", "her", "they", "we",
    "said", "with", "as", "for", "on", "at", "by", "from",
    "but", "not", "this", "be", "had", "have", "mr", "ms", "mrs",
}

def tokenize(text):
    text = text.lower()
    
    # Removing all punctuation 
    text = re.sub(r'[^\w\s]', '', text)
    
    words = text.split()
    
    # Removing stopwords
    words = [w for w in words if w not in stopwords]
    
    return words

tokenized_chapters = [tokenize(ch) for ch in chapters]

**Creating Table for each Term and Raw Word Count**

In [ ]:
# Getting vocabulary
vocab = set()
for chapter in tokenized_chapters:
    vocab.update(chapter)

vocab = sorted(vocab)

# Creating term-document Count Matrix
term_doc = pd.DataFrame(0, index=[f"Chapter_{i+1}" for i in range(len(tokenized_chapters))],
                        columns=vocab)

for i, chapter in enumerate(tokenized_chapters):
    counts = Counter(chapter)
    for word, count in counts.items():
        term_doc.iloc[i][word] = count

term_doc.head()

/var/folders/fm/1rc1d6s92qg_dtmxcn1__h1r0000gn/T/ipykernel_65078/1496975072.py:15: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  term_doc.iloc[i][word] = count


,1,12,1500,158th,17,1902,1906,1915,1919,1922,...,younger,youngerwith,your,youre,yours,yourself,youth,youve,youwhy,yukon
Chapter_1,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Chapter_2,0,0,0,0,0,0,0,1,0,0,...,2,0,1,0,0,0,0,1,0,0
Chapter_3,0,0,0,1,0,0,0,0,0,0,...,0,0,8,0,0,0,0,0,0,0
Chapter_4,0,0,0,0,0,0,0,0,0,0,...,0,0,7,5,0,2,0,1,0,0
Chapter_5,0,0,0,0,0,0,0,0,2,1,...,0,1,12,8,0,0,1,0,0,0


**Term Frequency Calculations**

In [82]:
tf = term_doc.div(term_doc.sum(axis=1), axis=0)

**Inverse Document Frequency Calculations**

In [83]:

N = len(term_doc)

doc_presence = (term_doc > 0).sum(axis=0)

idf = np.log(N / (1 + doc_presence))

In [84]:
idf = pd.Series(idf, index=term_doc.columns)

**TF-IDF Calculation**

In [85]:
tfidf = tf * idf

In [86]:
top_terms_per_chapter = {}

for chapter in tfidf.index:
    top_terms = tfidf.loc[chapter].sort_values(ascending=False).head(10)
    top_terms_per_chapter[chapter] = top_terms

# what was read in the tables was typing each chapter and taking each of the top words
# change Chapter_2 to any chapter from 1-9 (e.g. Chapter_3, or Chapter_8, etc)
# in same format to verify other chapter TF-IDF rankings
top_terms_per_chapter["Chapter_2"]

east         0.001653
miss         0.001324
boots        0.001302
longest      0.001302
candles      0.001302
mansion      0.001299
nick         0.001240
father       0.001236
baker        0.001204
balancing    0.000974
Name: Chapter_2, dtype: float64